In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/synapse_features/

/content/drive/.shortcut-targets-by-id/1FTGWdCLE0XRLzqrkRt55D9GnFIALowrJ/synapse_features


In [ ]:
!ls

features_flat.npy  idx_map.json  syn			      test_vol_h5
features.npy	   output	 Synapse_ActiveFT_10.0pc.txt  train_npz


In [ ]:
import os
path = "/content/drive/MyDrive/synapse_features/actft"
os.makedirs(path, exist_ok=True)

In [ ]:
%cd /content/

/content


In [ ]:
# --- Step 1: Clone repo and prepare data ---
!git clone https://github.com/Thinklab-SJTU/BiLAF

fatal: destination path 'BiLAF' already exists and is not an empty directory.


In [ ]:
%cd /content/BiLAF/data_selection/Core_Selection/

[Errno 2] No such file or directory: '/content/BiLAF/data_selection/Core_Selection/'
/content/drive/.shortcut-targets-by-id/1FTGWdCLE0XRLzqrkRt55D9GnFIALowrJ/synapse_features


In [ ]:
feature_path = "/content/drive/MyDrive/synapse_features/features_flat.npy"
id_path = "/content/drive/MyDrive/synapse_features/idx_map.json"
output_dir = "/content/drive/MyDrive/synapse_features/actft"

!python ActiveFT_ade20k.py \
  --feature_path $feature_path \
  --id_path $id_path \
  --output_dir $output_dir \
  --filename selected_points \
  --percent 10 \
  --init random \
  --distance cosine \
  --lr 0.001 \
  --max_iter 300 \
  --temperature 0.07



python3: can't open file '/content/drive/.shortcut-targets-by-id/1FTGWdCLE0XRLzqrkRt55D9GnFIALowrJ/synapse_features/ActiveFT_ade20k.py': [Errno 2] No such file or directory


In [ ]:
import json

# Load full filename list (implicit mapping)
input_path = '/content/drive/MyDrive/synapse_features/idx_map.json'
with open(input_path, 'r') as f:
    all_files = json.load(f)

# Create filename → index mapping
filename_to_idx = {name: i for i, name in enumerate(all_files)}
output_dir = "/content/drive/MyDrive/synapse_features/output/selected_points_ActiveFT_1.0pc.txt"
# Load the filenames selected by ActiveFT (core set)
with open(output_dir, 'r') as f:
    selected_files = [line.strip() for line in f.readlines()]

# Convert filenames to indices
core_indices = []
missing = []
for name in selected_files:
    if name in filename_to_idx:
        core_indices.append(filename_to_idx[name])
    else:
        missing.append(name)

# Save the output JSON (this is what density_cluster.py needs)selected_points_ActiveFT_1.0pc.txt
output_dir2 = "/content/drive/MyDrive/synapse_features/output/core_indices_1.0pc_syn.json"
with open(output_dir2, 'w') as f:
    json.dump(core_indices, f)

print(" Saved file: core_indices_10pc.json")
print(" Total converted:", len(core_indices))
if missing:
    print(" Missing filenames (not found in mapping):", missing[:10])


 Saved file: core_indices_10pc.json
 Total converted: 22


In [ ]:
%cd /content/


/content


In [ ]:
%cd /content/BiLAF/data_selection/Boundary_Selection/

/content/BiLAF/data_selection/Boundary_Selection


In [ ]:
!python /content/BiLAF/data_selection/Boundary_Selection/density_cluster.py \
  --features_inputs /content/drive/MyDrive/synapse_features/features_flat.npy \
  --indices_file_name /content/drive/MyDrive/synapse_features/output/core_indices_1.0pc_syn.json \
  --cur_number 22 \
  --budget 221 \
  --output_dir /content/drive/MyDrive/small_features/boundary_points_syn/

Load time: 2.5473
Step0 time: 10.3119
Step1 time: 66.2675
Step2 time: 59.7566
Total time: 138.8835
221


In [ ]:
import os

npz_dir = "/content/synapse/train_npz"
out_dir = "/content/synapse/lists_Synapse"

# get sorted file names
files = sorted([f for f in os.listdir(npz_dir) if f.endswith(".npz")])

# your BiLAF-selected indices
selected_indices = [...]  # your list

selected_files = [files[i] for i in selected_indices]

with open(out_dir + "/train.txt", "w") as f:
    for fn in selected_files:
        f.write(fn.replace(".npz", "") + "\n")


In [ ]:
!ls

core_indices_1.0pc_syn.json	  selected_points_ActiveFT_1.0pc.txt
Density_221_from_ActiveFT22.json


In [ ]:
%cd /content/drive/MyDrive/synapse_features/output/Density_221_from_ActiveFT22.json

[Errno 20] Not a directory: '/content/drive/MyDrive/synapse_features/output/Density_221_from_ActiveFT22.json'
/content/drive/.shortcut-targets-by-id/1FTGWdCLE0XRLzqrkRt55D9GnFIALowrJ/synapse_features/output


In [ ]:
!pwd

/content/drive/.shortcut-targets-by-id/1FTGWdCLE0XRLzqrkRt55D9GnFIALowrJ/synapse_features/output


In [ ]:
import os
import json
import shutil

# ============================================================
#                USER INPUT — CHANGE THESE PATHS
# ============================================================

# Folder containing ALL original NPZ slices
FULL_TRAIN_NPZ = "/content/drive/MyDrive/synapse_features/train_npz"

# Path to indices.json (BiLAF-selected indices)
INDICES_JSON = "/content/drive/MyDrive/synapse_features/output/Density_221_from_ActiveFT22.json"

# Path to ind_map.json (list of filenames indexed by ID)
IND_MAP_JSON = "/content/drive/MyDrive/synapse_features/idx_map.json"

# Final Synapse folder structure to create
ROOT = "/content/drive/MyDrive/synapse_features/syn"
TRAIN_NPZ = f"{ROOT}/train_npz"
TEST_VOL_H5 = f"{ROOT}/test_vol_h5"
LISTS = f"{ROOT}/lists_Synapse"

# Folder where your test .h5 volumes currently are
ORIGINAL_TEST_VOL = "/content/drive/MyDrive/synapse_features/test_vol_h5"

# ============================================================


# ============================================================
#        STEP 1 — Create the required directory structure
# ============================================================
os.makedirs(TRAIN_NPZ, exist_ok=True)
os.makedirs(TEST_VOL_H5, exist_ok=True)
os.makedirs(LISTS, exist_ok=True)

print("Created directory structure.")


# ============================================================
#        STEP 2 — Load indices and ind_map
# ============================================================
with open(INDICES_JSON, "r") as f:
    selected_indices = json.load(f)

with open(IND_MAP_JSON, "r") as f:
    ind_map = json.load(f)  # this must be a list

print(f"Loaded {len(selected_indices)} selected indices.")
print(f"Total indices available in ind_map: {len(ind_map)}")


# ============================================================
#        STEP 3 — Convert indices → filenames
# ============================================================
selected_files = [ind_map[i] for i in selected_indices]

print("Example selected filenames:")
print(selected_files[:10])


# ============================================================
#        STEP 4 — SAVE the selected filenames into a JSON file
# ============================================================
selected_json_path = f"{LISTS}/selected_files.json"

with open(selected_json_path, "w") as f:
    json.dump(selected_files, f, indent=2)

print(f"Saved selected filenames to: {selected_json_path}")


# ============================================================
#        STEP 5 — Copy only selected NPZ files to final folder
# ============================================================
missing = []

for fname in selected_files:
    src = os.path.join(FULL_TRAIN_NPZ, fname)
    dst = os.path.join(TRAIN_NPZ, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        missing.append(fname)

if missing:
    print("WARNING: Missing files:")
    print(missing)
else:
    print("All selected files copied successfully.")


# ============================================================
#        STEP 6 — Create train.txt
# ============================================================
train_txt_path = f"{LISTS}/train.txt"

with open(train_txt_path, "w") as f:
    for fname in selected_files:
        f.write(fname.replace(".npz", "") + "\n")

print(f"Created train.txt at: {train_txt_path}")


# ============================================================
#        STEP 7 — Copy test volumes + create test_vol.txt
# ============================================================
# copy .h5 files
for f in os.listdir(ORIGINAL_TEST_VOL):
    if f.endswith(".h5"):
        shutil.copy(os.path.join(ORIGINAL_TEST_VOL, f),
                    os.path.join(TEST_VOL_H5, f))

# create test_vol.txt
test_vol_txt = f"{LISTS}/test_vol.txt"
h5_files = sorted([x for x in os.listdir(TEST_VOL_H5) if x.endswith(".h5")])

with open(test_vol_txt, "w") as f:
    for h in h5_files:
        f.write(h.replace(".h5", "") + "\n")

print(f"Created test_vol.txt at: {test_vol_txt}")


# ============================================================
#                      DONE
# ============================================================
print("\nDataset building complete.")
print("Final structure:")
print(ROOT)
print(" ├── train_npz/   (filtered NPZ slices)")
print(" ├── test_vol_h5/ (test volumes)")
print(" └── lists_Synapse/")
print("       ├── train.txt")
print("       ├── test_vol.txt")
print("       └── selected_files.json")


Created directory structure.
Loaded 221 selected indices.
Total indices available in ind_map: 2211
Example selected filenames:
['case0005_slice039.npz', 'case0027_slice048.npz', 'case0026_slice065.npz', 'case0005_slice072.npz', 'case0005_slice004.npz', 'case0005_slice081.npz', 'case0005_slice048.npz', 'case0005_slice016.npz', 'case0005_slice062.npz', 'case0005_slice026.npz']
Saved selected filenames to: /content/drive/MyDrive/synapse_features/syn/lists_Synapse/selected_files.json
All selected files copied successfully.
Created train.txt at: /content/drive/MyDrive/synapse_features/syn/lists_Synapse/train.txt
Created test_vol.txt at: /content/drive/MyDrive/synapse_features/syn/lists_Synapse/test_vol.txt

Dataset building complete.
Final structure:
/content/drive/MyDrive/synapse_features/syn
 ├── train_npz/   (filtered NPZ slices)
 ├── test_vol_h5/ (test volumes)
 └── lists_Synapse/
       ├── train.txt
       ├── test_vol.txt
       └── selected_files.json


In [ ]:
%cd /content
!git clone https://github.com/JCruan519/VM-UNet.git


/content
fatal: destination path 'VM-UNet' already exists and is not an empty directory.


In [ ]:
%cd /content/VM-UNet/datasets
!touch __init__.py



/content/VM-UNet/datasets


In [ ]:
!ls

dataset.py  __init__.py  __pycache__


In [ ]:
%cd /content/VM-UNet/

/content/VM-UNet


In [ ]:
!git clone https://github.com/MzeroMiko/VMamba.git

Cloning into 'VMamba'...
remote: Enumerating objects: 8910, done.
remote: Counting objects: 100% (1398/1398), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 8910 (delta 1341), reused 1223 (delta 1223), pack-reused 7512 (from 2)
Receiving objects: 100% (8910/8910), 34.84 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (5492/5492), done.


In [ ]:
%cd VMamba/ops/
!ls


[Errno 2] No such file or directory: 'VMamba/ops/'
/content/VM-UNet
configs   engine.py	     models		  README.md  train_synapse.py
data	  engine_synapse.py  pre_trained_weights  results    utils.py
datasets  LICENSE	     __pycache__	  train.py   VMamba


In [ ]:
!python train_synapse.py

#----------Creating logger----------#
#----------GPU init----------#
#----------Preparing dataset----------#
#----------Prepareing Models----------#
Total model_dict: 222, Total pretrained_dict: 446, update: 117
Not loaded keys: ['layers.2.blocks.2.ln_1.weight', 'layers.2.blocks.2.ln_1.bias', 'layers.2.blocks.2.self_attention.x_proj_weight', 'layers.2.blocks.2.self_attention.dt_projs_weight', 'layers.2.blocks.2.self_attention.dt_projs_bias', 'layers.2.blocks.2.self_attention.A_logs', 'layers.2.blocks.2.self_attention.Ds', 'layers.2.blocks.2.self_attention.in_proj.weight', 'layers.2.blocks.2.self_attention.conv2d.weight', 'layers.2.blocks.2.self_attention.conv2d.bias', 'layers.2.blocks.2.self_attention.out_norm.weight', 'layers.2.blocks.2.self_attention.out_norm.bias', 'layers.2.blocks.2.self_attention.out_proj.weight', 'layers.2.blocks.3.ln_1.weight', 'layers.2.blocks.3.ln_1.bias', 'layers.2.blocks.3.self_attention.x_proj_weight', 'layers.2.blocks.3.self_attention.dt_projs_weight', 'la

In [ ]:
!pip install scikit-learn matplotlib thop h5py SimpleITK scikit-image medpy yacs


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=aab2b18e99ebe486bbfe9df3dbf7f666a2048cafed2aed6f84e1d4ec7aa75eb8
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


In [ ]:
!pip install ml_collections


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.1 MB/s eta 0:00:00


In [ ]:
# !pip install torch torchvision torchaudio

# !pip install packaging
# !pip install timm==0.4.12
# !pip install pytest chardet yacs termcolor
# !pip install submitit tensorboardX



!pip install mamba-ssm




  Using cached mamba_ssm-2.2.6.post3.tar.gz (113 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
^C


KeyboardInterrupt: 